# Notebook 1 (SVAMP) — SG Data Generation with Llama-3.2-3B-Instruct

**Dataset changed from ASDiv → SVAMP**
- SVAMP has ~1,000 grade-school math word problems designed to test robustness
- HuggingFace ID: `ChilleD/SVAMP`
- Problems use `Body` + `Question` fields (same pattern as ASDiv)
- Only changes from ASDiv version: Cell 4 (num_samples), Cell 5 (dataset loading), Cell 9 (test call), Cell 10 (field names + loop print)

**Expected valid rate: 70-85%**

**Memory: ~6GB float16 on P100 — safe without 4-bit**


In [1]:
# ── CELL 1: Install ───────────────────────────────────────────
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# ── CELL 2: HuggingFace Login ─────────────────────────────────
from huggingface_hub import login

login("")

In [3]:
# ── CELL 3: Imports + GPU Check ───────────────────────────────
import os, json, re, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

OUTPUT_DIR = "/kaggle/working/sg_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB


In [4]:
# ── CELL 4: Config ────────────────────────────────────────────
# num_samples: 50 = quick test | 900 = full run
# SVAMP has ~1,000 problems total — 900 is a safe full-run number

CONFIG = {
    "model_name"         : "Qwen/Qwen2.5-3B-Instruct",
    "num_samples"        : 900,       # ← change to 50 for a quick test first
    "random_seed"        : 42,
    "max_new_tokens"     : 150,
    "temperature"        : 0.1,
    "do_sample"          : True,
    "batch_size"         : 8,
    "max_words_per_step" : 20,
    "raw_output_file"    : f"{OUTPUT_DIR}/sg_raw.jsonl",
    "valid_output_file"  : f"{OUTPUT_DIR}/sg_valid.jsonl",
    "checkpoint_file"    : f"{OUTPUT_DIR}/checkpoint.json",
    "report_file"        : f"{OUTPUT_DIR}/generation_report.json",
    "save_every"         : 50,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")


Config:
  model_name            : Qwen/Qwen2.5-3B-Instruct
  num_samples           : 900
  random_seed           : 42
  max_new_tokens        : 150
  temperature           : 0.1
  do_sample             : True
  batch_size            : 8
  max_words_per_step    : 20
  raw_output_file       : /kaggle/working/sg_data/sg_raw.jsonl
  valid_output_file     : /kaggle/working/sg_data/sg_valid.jsonl
  checkpoint_file       : /kaggle/working/sg_data/checkpoint.json
  report_file           : /kaggle/working/sg_data/generation_report.json
  save_every            : 50


In [5]:
# ── CELL 5: Load SVAMP ───────────────────────────────────────
# SVAMP fields: ID, Body, Question, Equation, Answer, Type
# We combine Body + Question into one string (same shape as GSM8K's 'question').
# Answer is already a plain number (int or float) — no unit stripping needed.

print("Loading SVAMP from HuggingFace...")
svamp = load_dataset("ChilleD/SVAMP")

# SVAMP ships 'train' (~1,000) and 'test' splits — we use train
train_data = list(svamp["train"])

print(f"SVAMP total : {len(train_data)} problems")
print(f"Features    : {list(svamp['train'].features.keys())}")
print(f"\nRaw example:")
for k, v in train_data[0].items():
    print(f"  {k}: {v}")


def svamp_question(item):
    """Merge Body + Question into one string — equivalent to GSM8K 'question'."""
    return item["Body"].strip().rstrip(".") + " " + item["Question"].strip()


def svamp_answer(item):
    """
    Return a clean numeric string from SVAMP's Answer field.
    SVAMP answers are already plain numbers (e.g. 12, 3.5).
    We normalise 5.0 -> '5' for consistency.
    """
    try:
        f = float(item["Answer"])
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception:
        return str(item["Answer"]).strip()


random.seed(CONFIG["random_seed"])
num          = min(CONFIG["num_samples"], len(train_data))
indices      = random.sample(range(len(train_data)), num)
sampled_data = [train_data[i] for i in indices]

print(f"\nSampled    : {len(sampled_data)}")
print(f"\nExample question : {svamp_question(sampled_data[0])}")
print(f"Example answer   : {svamp_answer(sampled_data[0])}")


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ChilleD/SVAMP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading SVAMP from HuggingFace...


README.md:   0%|          | 0.00/675 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

SVAMP total : 700 problems
Features    : ['ID', 'Body', 'Question', 'Equation', 'Answer', 'Type', 'question_concat']

Raw example:
  ID: chal-777
  Body: There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups
  Question: How big is each group of bananas?
  Equation: ( 290.0 / 2.0 )
  Answer: 145
  Type: Common-Division
  question_concat: There are 87 oranges and 290 bananas in Philip's collection. If the bananas are organized into 2 groups and oranges are organized into 93 groups How big is each group of bananas?

Sampled    : 700

Example question : 46 campers went rowing on a day. 43 campers went rowing in the morning and some more campers went rowing in the afternoon How many campers went rowing in the afternoon?
Example answer   : 3


In [6]:
# ── CELL 6: Load Qwen2.5-3B-Instruct ────────────────────────
print(f"Loading {CONFIG['model_name']}...")
print("Expected GPU memory: ~6GB in float16")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code = True,
)
tokenizer.padding_side = "left"    # critical for batch generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    dtype      = torch.float16,
    device_map = "auto",
)
model.eval()

used_gb  = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nModel loaded!")
print(f"GPU used  : {used_gb:.2f} GB / {total_gb:.1f} GB")
print(f"Headroom  : {total_gb - used_gb:.1f} GB remaining")
print(f"Padding   : {tokenizer.padding_side} ✅")

if used_gb > 10:
    print("⚠️ WARNING: High memory. Reduce batch_size to 4 in Config if OOM.")
else:
    print("✅ Memory looks good.")

Loading Qwen/Qwen2.5-3B-Instruct...
Expected GPU memory: ~6GB in float16


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded!
GPU used  : 3.09 GB / 15.6 GB
Headroom  : 12.5 GB remaining
Padding   : left ✅
✅ Memory looks good.


In [7]:
# ── CELL 7: Prompt ────────────────────────────────────────────
# Identical to ASDiv version — prompt format does not depend on dataset.

SYSTEM_PROMPT = """You are a math problem decomposition assistant.
Your job: break a math problem into 2-5 numbered solution steps.

RULES:
1. Use format: Step 1: ... Step 2: ... etc.
2. Each step must be under 15 words.
3. Do NOT perform calculations or write numbers from computation.
4. Do NOT write the final answer.
5. Write ONLY the steps. Stop immediately after the last step.

EXAMPLE:
Problem: John earns $10/hour and works 8 hours. What does he earn?
Step 1: Identify the hourly rate and total hours worked.
Step 2: Multiply the hourly rate by the number of hours.
Step 3: The result is the total earnings."""

USER_TEMPLATE = """Problem: {question}"""


def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_TEMPLATE.format(question=question)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
    )


print("Prompt preview:")
print(build_prompt(svamp_question(sampled_data[0])))


Prompt preview:
<|im_start|>system
You are a math problem decomposition assistant.
Your job: break a math problem into 2-5 numbered solution steps.

RULES:
1. Use format: Step 1: ... Step 2: ... etc.
2. Each step must be under 15 words.
3. Do NOT perform calculations or write numbers from computation.
4. Do NOT write the final answer.
5. Write ONLY the steps. Stop immediately after the last step.

EXAMPLE:
Problem: John earns $10/hour and works 8 hours. What does he earn?
Step 1: Identify the hourly rate and total hours worked.
Step 2: Multiply the hourly rate by the number of hours.
Step 3: The result is the total earnings.<|im_end|>
<|im_start|>user
Problem: 46 campers went rowing on a day. 43 campers went rowing in the morning and some more campers went rowing in the afternoon How many campers went rowing in the afternoon?<|im_end|>
<|im_start|>assistant



In [8]:
# ── CELL 8: Cleaning + Filtering ─────────────────────────────
# Identical to GSM8K version.

def clean_sg_output(text: str) -> str:
    """Extract only Step lines. Stop at first non-step line."""
    lines        = text.strip().split("\n")
    clean_lines  = []
    step_started = False

    for line in lines:
        line = line.strip()
        if not line:
            continue
        is_step = bool(re.match(
            r"^(Step\s*\d+[:.)]|\d+[.)]\s)", line, re.IGNORECASE
        ))
        if is_step:
            step_started = True
            trimmed = re.split(r"\.\s+[A-Z]", line)[0]
            if not trimmed.endswith("."):
                trimmed += "."
            clean_lines.append(trimmed)
        elif step_started:
            break

    return "\n".join(clean_lines)


def extract_steps(text: str) -> list:
    return [
        l.strip() for l in text.strip().split("\n")
        if re.match(r"^(Step\s*\d+[:.)]|\d+[.)]\s)", l.strip(), re.IGNORECASE)
    ]


def step_body(step: str) -> str:
    return re.sub(
        r"^(Step\s*\d+[:.)]|\d+[.)]\s)", "", step, flags=re.IGNORECASE
    ).strip()


def contains_calculation(text: str) -> bool:
    patterns = [
        r"\d+\s*[\+\-\×\÷\*\/]\s*\d+",
        r"=\s*\$?\d+",
        r"the answer is\s+\d+",
        r"\$\s*\d+\.?\d*",
        r"\\frac", r"\\times", r"\\left",
        r"\\\(", r"\\\)",
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


def steps_sequential(steps: list) -> bool:
    for i, step in enumerate(steps):
        m = re.match(r"^(?:Step\s*)(\d+)", step, re.IGNORECASE)
        if m and int(m.group(1)) != i + 1:
            return False
    return True


def steps_concise(steps: list, max_words: int) -> bool:
    return all(len(step_body(s).split()) <= max_words for s in steps)


def is_valid_sg(text: str) -> tuple:
    if not text or len(text.strip()) < 15:
        return False, "too_short"

    steps = extract_steps(text)

    if len(steps) < 2:                                        return False, "too_few_steps"
    if len(steps) > 5:                                        return False, "too_many_steps"
    if contains_calculation(text):                            return False, "contains_calculation"
    if not steps_sequential(steps):                          return False, "non_sequential_steps"
    if not steps_concise(steps, CONFIG["max_words_per_step"]): return False, "step_too_long"

    action_verbs = [
        "calculate", "determine", "identify", "find", "compute",
        "add", "subtract", "multiply", "divide", "sum", "count",
        "check", "compare", "convert", "evaluate", "use", "total",
        "measure", "estimate", "figure", "apply"
    ]
    if not any(v in text.lower() for v in action_verbs):
        return False, "no_action_verb"

    return True, "ok"


print("Filters loaded. Running sanity check...")
good = """Step 1: Identify the number of items in each group.
Step 2: Multiply the number of groups by items per group.
Step 3: Add any remaining items to get the total."""
v, r = is_valid_sg(good)
steps = extract_steps(good)
print(f"Test SG → valid={v}, reason={r}, steps={len(steps)}")
for s in steps:
    print(f"  [{len(step_body(s).split()):2d}w] {s}")

Filters loaded. Running sanity check...
Test SG → valid=True, reason=ok, steps=3
  [ 8w] Step 1: Identify the number of items in each group.
  [ 9w] Step 2: Multiply the number of groups by items per group.
  [ 8w] Step 3: Add any remaining items to get the total.


In [9]:
# ── CELL 9: Generation + Single Test ─────────────────────────

def generate_sg_batch(questions: list) -> list:
    prompts = [build_prompt(q) for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = CONFIG["max_new_tokens"],
            temperature        = CONFIG["temperature"],
            do_sample          = CONFIG["do_sample"],
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.2,
        )

    input_len = inputs["input_ids"].shape[1]
    results   = []
    for output in outputs:
        raw   = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
        clean = clean_sg_output(raw)
        results.append((raw, clean))
    return results


# ── Single test ──
print("=" * 55)
print("SINGLE SAMPLE TEST")
print("=" * 55)

test_q   = svamp_question(sampled_data[0])
test_out = generate_sg_batch([test_q])
raw, clean = test_out[0]

print(f"Question:\n{test_q}")
print(f"\nRaw output:\n{raw}")
print(f"\nCleaned SG:\n{clean}")

steps = extract_steps(clean)
valid, reason = is_valid_sg(clean)
print(f"\nResult → valid={valid} | reason={reason} | steps={len(steps)}")
for s in steps:
    words = len(step_body(s).split())
    flag  = "✅" if words <= CONFIG["max_words_per_step"] else "❌ TOO LONG"
    print(f"  [{words:2d}w] {flag}  {s}")

print("\n" + "=" * 55)
print("If steps look clean and valid=True → run Cell 10")
print("=" * 55)


SINGLE SAMPLE TEST
Question:
46 campers went rowing on a day. 43 campers went rowing in the morning and some more campers went rowing in the afternoon How many campers went rowing in the afternoon?

Raw output:
Step 1: Total campers who went rowing equals those in the morning plus those in the afternoon.
Step 2: Subtract the morning attendees from the total to find afternoon participants.
Step 3: Calculate the difference between total and morning attendance for afternoon count.
Step 4: Afternoon participant figure will complete the equation given above.

Cleaned SG:
Step 1: Total campers who went rowing equals those in the morning plus those in the afternoon.
Step 2: Subtract the morning attendees from the total to find afternoon participants.
Step 3: Calculate the difference between total and morning attendance for afternoon count.
Step 4: Afternoon participant figure will complete the equation given above.

Result → valid=True | reason=ok | steps=4
  [15w] ✅  Step 1: Total campers wh

In [10]:
# ── CELL 10: Main Loop ────────────────────────────────────────
# Only change from ASDiv version: use svamp_question() / svamp_answer()
# instead of asdiv_question() / asdiv_answer().

print(f"Generating SG for {len(sampled_data)} SVAMP questions...")
print("-" * 50)

results   = []
start_idx = 0

# Resume from checkpoint if session died
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["raw_output_file"]):
        with open(CONFIG["raw_output_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed: idx={start_idx}, saved={len(results)}")
else:
    print("Starting fresh...")

for batch_start in tqdm(
    range(start_idx, len(sampled_data), CONFIG["batch_size"]),
    desc="Generating"
):
    batch_end = min(batch_start + CONFIG["batch_size"], len(sampled_data))
    batch     = sampled_data[batch_start:batch_end]

    # ── SVAMP-specific: build question strings and extract clean answers ──
    questions = [svamp_question(b) for b in batch]
    answers   = [svamp_answer(b)   for b in batch]

    try:
        batch_out = generate_sg_batch(questions)
    except RuntimeError as e:
        print(f"OOM at batch {batch_start} — try reducing batch_size in Config")
        print(f"Error: {e}")
        break

    for i, (question, answer, (sg_raw, sg_clean)) in enumerate(
        zip(questions, answers, batch_out)
    ):
        valid, reason = is_valid_sg(sg_clean)
        steps         = extract_steps(sg_clean)

        results.append({
            "id"        : batch_start + i,
            "question"  : question,
            "sg_raw"    : sg_raw,
            "sg_clean"  : sg_clean,
            "sg_steps"  : steps,
            "gt_answer" : answer,   # already a clean numeric string
            "valid"     : valid,
            "reason"    : reason,
        })

    # Checkpoint
    if len(results) % CONFIG["save_every"] < CONFIG["batch_size"]:
        with open(CONFIG["raw_output_file"], "w") as f:
            for item in results: f.write(json.dumps(item) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": batch_end}, f)

# Final save
with open(CONFIG["raw_output_file"], "w") as f:
    for item in results: f.write(json.dumps(item) + "\n")

valid_count = sum(1 for r in results if r["valid"])
print(f"\nTotal     : {len(results)}")
print(f"Valid     : {valid_count}")
print(f"Invalid   : {len(results) - valid_count}")
print(f"Valid rate: {valid_count/max(1,len(results))*100:.1f}%")


Generating SG for 700 SVAMP questions...
--------------------------------------------------
Starting fresh...



Generating: 100%|██████████| 88/88 [09:31<00:00,  6.50s/it]


Total     : 700
Valid     : 688
Invalid   : 12
Valid rate: 98.3%


In [11]:
# ── CELL 11: Save + Show Samples ─────────────────────────────

valid_results = [r for r in results if r["valid"]]

with open(CONFIG["valid_output_file"], "w") as f:
    for item in valid_results:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(valid_results)} valid SG → {CONFIG['valid_output_file']}")
print("\n" + "=" * 60)
print("SAMPLE VALID SG OUTPUTS")
print("=" * 60)

for i, item in enumerate(valid_results[:5]):
    print(f"\n--- Example {i+1} ---")
    print(f"Q : {item['question'][:100]}")
    print("SG:")
    for s in item["sg_steps"]:
        words = len(step_body(s).split())
        print(f"  [{words:2d}w] {s}")
    print(f"GT: {item['gt_answer']}")

Saved 688 valid SG → /kaggle/working/sg_data/sg_valid.jsonl

SAMPLE VALID SG OUTPUTS

--- Example 1 ---
Q : 46 campers went rowing on a day. 43 campers went rowing in the morning and some more campers went ro
SG:
  [15w] Step 1: Total campers who went rowing equals those in the morning plus those in the afternoon.
  [11w] Step 2: Subtract the morning attendees from the total to find afternoon participants.
  [11w] Step 3: Calculate the difference between total and morning attendance for afternoon count.
  [ 9w] Step 4: Determine how many were present during the afternoon session.
  [10w] Step 5: Finish calculation to get the exact number of afternoon attendees.
GT: 3

--- Example 2 ---
Q : Randy has 37 blocks. He uses 33 blocks to build a tower and 13 blocks to build a house How many more
SG:
  [ 9w] Step 1: Determine how many blocks were used for both structures.
  [13w] Step 2: Subtract the blocks used for the house from those used for the tower.
  [10w] Step 3: Calculate the differe